# Chinese Understanding Benchmark: Qwen 4B vs Gemma E2B vs Gemma E4B 4-bit MLX

This notebook evaluates Chinese understanding on CLUE tasks using:

1. `Qwen/Qwen3-4B-Instruct-2507` via Transformers
2. `google/gemma-4-E2B-it` via Transformers
3. `mlx-community/gemma-4-e4b-it-OptiQ-4bit` via MLX-LM

**v8 fixes:** based on the working v6 notebook; each model loads once across all tasks, previous model memory is cleared before the next model loads, Gemma E4B 4-bit uses a separate MLX backend, AFQMC/CMNLI label mappings stay fixed, and TNEWS mapping is handled safely.


In [1]:
# Optional install cell. Run only if your environment is missing packages.
# Recommended: install from terminal, then restart the Jupyter kernel.
# If zsh complains about brackets, keep quotes around any extras.
# If torchaudio causes a torch-version conflict and you do not need audio, uninstall it.

!pip install -U transformers datasets accelerate peft trl scikit-learn pandas tqdm sentencepiece mlx-lm
!pip uninstall -y torchvision torchaudio


Found existing installation: torchaudio 2.6.0
Uninstalling torchaudio-2.6.0:
  Successfully uninstalled torchaudio-2.6.0


In [2]:
import os
import re
import time
import gc
from typing import Dict, List, Any, Optional, Tuple

import pandas as pd
import torch
from tqdm.auto import tqdm
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score

from transformers import AutoTokenizer, AutoModelForCausalLM

try:
    from mlx_lm import load as mlx_load
    from mlx_lm import generate as mlx_generate
    MLX_AVAILABLE = True
except Exception as e:
    MLX_AVAILABLE = False
    MLX_IMPORT_ERROR = repr(e)

try:
    from transformers import TrainingArguments
    from peft import LoraConfig, PeftModel
    from trl import SFTTrainer
    PEFT_AVAILABLE = True
except Exception as e:
    PEFT_AVAILABLE = False
    PEFT_IMPORT_ERROR = repr(e)

print('torch:', torch.__version__)
import transformers
print('transformers:', transformers.__version__)
print('mlx-lm available:', MLX_AVAILABLE)
if not MLX_AVAILABLE:
    print('MLX import error:', MLX_IMPORT_ERROR)
print('peft/trl available:', PEFT_AVAILABLE)
if not PEFT_AVAILABLE:
    print('PEFT import error:', PEFT_IMPORT_ERROR)


torch: 2.12.0
transformers: 5.8.1
mlx-lm available: True
peft/trl available: True


In [3]:
# Main configuration

# backend:
# - 'transformers': regular Hugging Face Transformers loading
# - 'mlx': Apple MLX-LM loading, useful for quantized MLX checkpoints on macOS
MODELS = {
    'qwen3_4b_instruct_2507': {
        'model_id': 'Qwen/Qwen3-4B-Instruct-2507',
        'backend': 'transformers',
    },
    'gemma_e2b_it': {
        'model_id': 'google/gemma-4-E2B-it',
        'backend': 'transformers',
    },
    'gemma_e4b_it_4bit_mlx': {
        'model_id': 'mlx-community/gemma-4-e4b-it-OptiQ-4bit',
        'backend': 'mlx',
    },
}

TASKS = ['afqmc', 'tnews', 'cmnli']
SPLIT = 'validation'
MAX_SAMPLES = 200
RESULTS_DIR = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)

# For Mac/CPU/MPS setups, use auto device placement. For CUDA machines, this also works.
# If a Transformers model becomes extremely slow and prints disk-offload warnings,
# reduce model size or use a quantized/MLX checkpoint.
DEVICE_MAP = 'auto'
TORCH_DTYPE = torch.float16


In [4]:
# Label specifications.
# We score dataset IDs, but ask models to output stable Chinese label names.
# Important: do NOT blindly overwrite AFQMC/CMNLI from dataset.features, because some CLUE
# versions expose feature names as raw IDs like "0"/"1" rather than semantic labels.
# TNEWS can vary by dataset version, so we infer it carefully when useful.

TASK_SPECS = {
    'afqmc': {
        'label_id_to_name': {0: '不同', 1: '相同'},
        'label_name_to_id': {'不同': '0', '相同': '1'},
    },
    'cmnli': {
        # In the Hugging Face CLUE CMNLI validation split observed here:
        # 0 = neutral, 1 = entailment, 2 = contradiction
        'label_id_to_name': {0: '中立', 1: '蕴含', 2: '矛盾'},
        'label_name_to_id': {'中立': '0', '蕴含': '1', '矛盾': '2'},
    },
    'tnews': {
        # Hugging Face clue/tnews often uses contiguous IDs 0-14 in this order.
        'label_id_to_name': {
            0: '故事', 1: '文化', 2: '娱乐', 3: '体育', 4: '财经',
            5: '房产', 6: '汽车', 7: '教育', 8: '科技', 9: '国际',
            10: '股票', 11: '旅游', 12: '军事', 13: '农业', 14: '电竞',
        },
        'label_name_to_id': {
            '故事': '0', '文化': '1', '娱乐': '2', '体育': '3', '财经': '4',
            '房产': '5', '汽车': '6', '教育': '7', '科技': '8', '国际': '9',
            '股票': '10', '旅游': '11', '军事': '12', '农业': '13', '电竞': '14',
        },
    },
}

LABEL_ALIASES = {
    'afqmc': {
        '相同': ['相同', '一致', '等价', '同义', '语义相同', '是', 'yes', 'true', '1'],
        '不同': ['不同', '不相同', '不一致', '不等价', '语义不同', '否', 'no', 'false', '0'],
    },
    'cmnli': {
        '中立': ['中立', '无关', '无法判断', 'neutral', '0'],
        '蕴含': ['蕴含', '包含', '推出', 'entailment', 'entails', '1'],
        '矛盾': ['矛盾', '冲突', 'contradiction', 'contradict', '2'],
    },
    'tnews': {
        '故事': ['故事', '0', '100'],
        '文化': ['文化', '1', '101'],
        '娱乐': ['娱乐', '2', '102'],
        '体育': ['体育', '3', '103'],
        '财经': ['财经', '4', '104'],
        '房产': ['房产', '5', '106'],
        '汽车': ['汽车', '6', '107'],
        '教育': ['教育', '7', '108'],
        '科技': ['科技', '8', '109'],
        '国际': ['国际', '9', '113'],
        '股票': ['股票', '10', '114'],
        '旅游': ['旅游', '11', '112'],
        '军事': ['军事', '12', '110'],
        '农业': ['农业', '13', '115'],
        '电竞': ['电竞', '14', '116'],
    },
}

TNEWS_NAME_ZH = {
    'news_story': '故事', 'story': '故事', '100': '故事',
    'news_culture': '文化', 'culture': '文化', '101': '文化',
    'news_entertainment': '娱乐', 'entertainment': '娱乐', '102': '娱乐',
    'news_sports': '体育', 'sports': '体育', '103': '体育',
    'news_finance': '财经', 'finance': '财经', '104': '财经',
    'news_house': '房产', 'house': '房产', 'real_estate': '房产', '106': '房产',
    'news_car': '汽车', 'car': '汽车', 'auto': '汽车', '107': '汽车',
    'news_edu': '教育', 'education': '教育', 'edu': '教育', '108': '教育',
    'news_tech': '科技', 'technology': '科技', 'tech': '科技', '109': '科技',
    'news_military': '军事', 'military': '军事', '110': '军事',
    'news_travel': '旅游', 'travel': '旅游', '112': '旅游',
    'news_world': '国际', 'world': '国际', 'international': '国际', '113': '国际',
    'news_stock': '股票', 'stock': '股票', '114': '股票',
    'news_agriculture': '农业', 'agriculture': '农业', '115': '农业',
    'news_game': '电竞', 'game': '电竞', 'esports': '电竞', '116': '电竞',
}


def update_task_spec_from_dataset(task: str, dataset) -> str:
    """Update only TNEWS when dataset feature names are semantic.

    AFQMC/CMNLI are kept fixed because their ClassLabel names can be raw string IDs,
    which caused previous all-invalid scoring.
    """
    if task != 'tnews':
        return 'fixed_manual'

    label_feature = dataset.features.get('label')
    names = getattr(label_feature, 'names', None)
    if not names:
        return 'manual_fallback'

    # If the dataset exposes old CLUE IDs such as 100/101/102, remap them to current 0-14 IDs.
    canonical_names = [TNEWS_NAME_ZH.get(str(name).strip(), str(name).strip()) for name in names]

    # If feature names are just 0/1/2/... then they are not semantic; keep the manual fallback.
    if all(str(x).isdigit() for x in canonical_names):
        return 'manual_fallback_numeric_features'

    TASK_SPECS['tnews'] = {
        'label_id_to_name': {i: name for i, name in enumerate(canonical_names)},
        'label_name_to_id': {name: str(i) for i, name in enumerate(canonical_names)},
    }
    LABEL_ALIASES['tnews'] = {
        canonical: sorted(set([canonical, str(original), str(i)]))
        for i, (original, canonical) in enumerate(zip(names, canonical_names))
    }
    return 'dataset_features_tnews'


def get_label_names(task: str) -> List[str]:
    return list(TASK_SPECS[task]['label_name_to_id'].keys())


def label_id_to_name(task: str, label_id: Any) -> str:
    try:
        key = int(label_id)
    except Exception:
        return str(label_id)
    return TASK_SPECS[task]['label_id_to_name'].get(key, str(label_id))


def label_name_to_id(task: str, label_name: str) -> str:
    return TASK_SPECS[task]['label_name_to_id'].get(label_name, '__invalid__')


In [5]:
def build_prompt(task: str, example: Dict[str, Any]) -> str:
    labels = '、'.join(get_label_names(task))

    if task == 'afqmc':
        return f"""你是中文二分类器。只输出“相同”或“不同”其中一个词，不要解释。

句子1：{example['sentence1']}
句子2：{example['sentence2']}

语义是否相同？答案："""

    if task == 'cmnli':
        return f"""你是中文自然语言推理分类器。只能输出一个标签，不要解释。

任务：判断“假设”与“前提”的关系。
可选标签：{labels}

前提：{example['sentence1']}
假设：{example['sentence2']}

答案："""

    if task == 'tnews':
        return f"""你是中文新闻标题分类器。只能输出一个标签，不要解释。

任务：判断新闻标题所属类别。
可选标签：{labels}

标题：{example['sentence']}

答案："""

    raise ValueError(f'Unsupported task: {task}')


def normalize_output(text: Any) -> str:
    text = str(text).strip().lower()
    for prefix in ['答案：', '答案:', '标签：', '标签:', '类别：', '类别:']:
        text = text.replace(prefix, '')
    text = text.replace('\n', ' ')
    text = re.sub(r'''[。，“”，、；;:：\[\]\(\)（）\"']''', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def extract_label_name(task: str, text: Any) -> str:
    text_norm = normalize_output(text)

    # Exact or alias match first.
    for canonical_label, aliases in LABEL_ALIASES.get(task, {}).items():
        for alias in aliases:
            alias_norm = normalize_output(alias)
            if text_norm == alias_norm:
                return canonical_label

    # Exact canonical label match.
    for label in get_label_names(task):
        if text_norm == normalize_output(label):
            return label

    # Substring match. This catches outputs like "'娱乐'" or "答案是：娱乐".
    for canonical_label, aliases in LABEL_ALIASES.get(task, {}).items():
        for alias in aliases:
            alias_norm = normalize_output(alias)
            if alias_norm and alias_norm in text_norm:
                return canonical_label

    for label in get_label_names(task):
        label_norm = normalize_output(label)
        if label_norm and label_norm in text_norm:
            return label

    # Last resort: extract first standalone integer label ID.
    digit_match = re.search(r'\b\d+\b', text_norm)
    if digit_match:
        digit = digit_match.group(0)
        for canonical_label, aliases in LABEL_ALIASES.get(task, {}).items():
            if digit in [str(x) for x in aliases]:
                return canonical_label
        for label_name, label_id in TASK_SPECS[task]['label_name_to_id'].items():
            if digit == str(label_id):
                return label_name

    return '__invalid__'


def format_example_for_sft(task: str, ex: Dict[str, Any]) -> str:
    prompt = build_prompt(task, ex)
    answer = label_id_to_name(task, ex['label'])
    return prompt + answer


In [6]:
def cleanup_model(model=None, tokenizer=None):
    """Aggressively release model memory between models.

    This is especially important on Apple Silicon. It is still not as strong as restarting
    the Jupyter kernel, but it prevents keeping Qwen in memory while Gemma loads.
    """
    try:
        if model is not None and hasattr(model, 'cpu'):
            model.cpu()
    except Exception:
        pass
    try:
        del model
    except Exception:
        pass
    try:
        del tokenizer
    except Exception:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    try:
        torch.mps.empty_cache()
    except Exception:
        pass


def get_model_id(model_cfg_or_id):
    if isinstance(model_cfg_or_id, dict):
        return model_cfg_or_id['model_id']
    return model_cfg_or_id


def get_backend(model_cfg_or_id):
    if isinstance(model_cfg_or_id, dict):
        return model_cfg_or_id.get('backend', 'transformers')
    return 'transformers'


def load_model_and_tokenizer(model_id: str, backend: str = 'transformers'):
    cleanup_model()

    if backend == 'mlx':
        if not MLX_AVAILABLE:
            raise RuntimeError(f'mlx-lm is not available: {MLX_IMPORT_ERROR}')
        model, tokenizer = mlx_load(model_id)
        return tokenizer, model

    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=TORCH_DTYPE,
        device_map=DEVICE_MAP,
        trust_remote_code=True,
    )
    model.eval()
    return tokenizer, model


def apply_chat_template_if_available(tokenizer, prompt: str) -> str:
    # For instruction-tuned chat models, use the tokenizer chat template when available.
    if getattr(tokenizer, 'chat_template', None):
        messages = [{'role': 'user', 'content': prompt}]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return prompt


@torch.no_grad()
def generate_answer_transformers(tokenizer, model, prompt: str, max_new_tokens: int = 4) -> str:
    text = apply_chat_template_if_available(tokenizer, prompt)
    inputs = tokenizer(text, return_tensors='pt')
    try:
        inputs = inputs.to(model.device)
    except Exception:
        # Some accelerated/device_map models do not have a simple single device.
        pass

    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(output[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()


def generate_answer_mlx(tokenizer, model, prompt: str, max_new_tokens: int = 4) -> str:
    text = apply_chat_template_if_available(tokenizer, prompt)
    # mlx_lm.generate returns generated text. verbose=False prevents token streaming/noisy output.
    try:
        return mlx_generate(model, tokenizer, prompt=text, max_tokens=max_new_tokens, verbose=False).strip()
    except TypeError:
        # Older mlx-lm versions may not support verbose.
        return mlx_generate(model, tokenizer, prompt=text, max_tokens=max_new_tokens).strip()


def generate_answer(tokenizer, model, prompt: str, max_new_tokens: int = 4, backend: str = 'transformers') -> str:
    if backend == 'mlx':
        return generate_answer_mlx(tokenizer, model, prompt, max_new_tokens=max_new_tokens)
    return generate_answer_transformers(tokenizer, model, prompt, max_new_tokens=max_new_tokens)


In [9]:
def evaluate_loaded_model_on_task(
    model_key: str,
    model_id: str,
    tokenizer,
    model,
    task: str,
    backend: str = 'transformers',
    split: str = 'validation',
    max_samples: Optional[int] = 200,
    debug_first_n: int = 5,
):
    dataset = load_dataset('clue', task, split=split)
    label_id_style = update_task_spec_from_dataset(task, dataset)

    if max_samples is not None:
        dataset = dataset.select(range(min(max_samples, len(dataset))))

    print(f'{task} labels ({label_id_style}):', TASK_SPECS[task]['label_id_to_name'])

    rows = []
    y_true, y_pred = [], []
    start = time.time()

    for i, ex in enumerate(tqdm(dataset, desc=f'{model_key}/{task}')):
        prompt = build_prompt(task, ex)
        raw = generate_answer(tokenizer, model, prompt, max_new_tokens=4, backend=backend)
        pred_name = extract_label_name(task, raw)
        pred_id = label_name_to_id(task, pred_name)
        gold_id = str(ex['label'])
        gold_name = label_id_to_name(task, gold_id)

        y_true.append(gold_id)
        y_pred.append(pred_id)
        row = {
            'model_key': model_key,
            'model_id': model_id,
            'backend': backend,
            'task': task,
            'gold_id': gold_id,
            'gold_name': gold_name,
            'raw': repr(raw),
            'pred_name': pred_name,
            'pred_id': pred_id,
        }
        rows.append(row)
        if i < debug_first_n:
            print(row)

    elapsed = time.time() - start
    invalid_rate = sum(p == '__invalid__' for p in y_pred) / len(y_pred)
    summary = {
        'model_key': model_key,
        'model_id': model_id,
        'backend': backend,
        'task': task,
        'split': split,
        'samples': len(y_true),
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'invalid_rate': invalid_rate,
        'seconds': elapsed,
        'samples_per_second': len(y_true) / elapsed if elapsed > 0 else None,
        'label_id_style': label_id_style,
    }
    return summary, pd.DataFrame(rows)


In [11]:
def evaluate_model_all_tasks(
    model_key: str,
    model_cfg_or_id,
    tasks: List[str],
    split: str = 'validation',
    max_samples: Optional[int] = 200,
    debug_first_n: int = 3,
):
    """Load one model once, evaluate all tasks, then clear it.

    This avoids reloading Gemma/Qwen for every task and prevents keeping the previous model
    resident while the next one loads.
    """
    model_id = get_model_id(model_cfg_or_id)
    backend = get_backend(model_cfg_or_id)

    print(f'Loading model once: {model_key} -> {model_id} [{backend}]')
    tokenizer, model = load_model_and_tokenizer(model_id, backend=backend)
    summaries, pred_dfs = [], []
    try:
        for task in tasks:
            summary, pred_df = evaluate_loaded_model_on_task(
                model_key=model_key,
                model_id=model_id,
                tokenizer=tokenizer,
                model=model,
                task=task,
                backend=backend,
                split=split,
                max_samples=max_samples,
                debug_first_n=debug_first_n,
            )
            print(summary)
            summaries.append(summary)
            pred_dfs.append(pred_df)
    finally:
        print(f'Clearing model from memory: {model_key}')
        cleanup_model(model, tokenizer)
    return summaries, pred_dfs


In [12]:
# Run baseline evaluation.
# Each model is loaded once, evaluated on all tasks, then cleared before the next model loads.
# Note: Gemma E4B 4-bit uses MLX-LM, while Qwen/Gemma E2B use Transformers.

all_summaries = []
all_predictions = []

for model_key, model_cfg in MODELS.items():
    summaries, pred_dfs = evaluate_model_all_tasks(
        model_key=model_key,
        model_cfg_or_id=model_cfg,
        tasks=TASKS,
        split=SPLIT,
        max_samples=MAX_SAMPLES,
        debug_first_n=3,
    )
    all_summaries.extend(summaries)
    all_predictions.extend(pred_dfs)

summary_df = pd.DataFrame(all_summaries)
predictions_df = pd.concat(all_predictions, ignore_index=True)
summary_df.to_csv(os.path.join(RESULTS_DIR, 'baseline_summary.csv'), index=False)
predictions_df.to_csv(os.path.join(RESULTS_DIR, 'baseline_predictions.csv'), index=False)
summary_df


Loading model once: qwen3_4b_instruct_2507 -> Qwen/Qwen3-4B-Instruct-2507 [transformers]


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

afqmc labels (fixed_manual): {0: '不同', 1: '相同'}


qwen3_4b_instruct_2507/afqmc:   0%|          | 0/200 [00:00<?, ?it/s]

...

{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'backend': 'transformers', 'task': 'tnews', 'gold_id': '2', 'gold_name': '娱乐', 'raw': "'娱乐'", 'pred_name': '娱乐', 'pred_id': '2'}
{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'backend': 'transformers', 'task': 'tnews', 'gold_id': '9', 'gold_name': '军事', 'raw': "'军事'", 'pred_name': '军事', 'pred_id': '9'}
{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'backend': 'transformers', 'task': 'tnews', 'gold_id': '4', 'gold_name': '财经', 'raw': "'财经'", 'pred_name': '财经', 'pred_id': '4'}
{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'backend': 'transformers', 'task': 'tnews', 'split': 'validation', 'samples': 200, 'accuracy': 0.465, 'macro_f1': 0.3986677421980278, 'invalid_rate': 0.06, 'seconds': 77.61896777153015, 'samples_per_second': 2.576689767231844, 'label_id_style': 'dataset_features_tnews'}
cmnli labels (fixed_manual): {0: '中立', 1: '蕴含', 2: '矛盾'}


gemma_e2b_it/cmnli:   0%|          | 0/200 [00:00<?, ?it/s]

{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'backend': 'transformers', 'task': 'cmnli', 'gold_id': '0', 'gold_name': '中立', 'raw': "'中立'", 'pred_name': '中立', 'pred_id': '0'}
{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'backend': 'transformers', 'task': 'cmnli', 'gold_id': '1', 'gold_name': '蕴含', 'raw': "'蕴含'", 'pred_name': '蕴含', 'pred_id': '1'}
{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'backend': 'transformers', 'task': 'cmnli', 'gold_id': '2', 'gold_name': '矛盾', 'raw': "'矛盾'", 'pred_name': '矛盾', 'pred_id': '2'}
{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'backend': 'transformers', 'task': 'cmnli', 'split': 'validation', 'samples': 200, 'accuracy': 0.64, 'macro_f1': 0.6081352729236104, 'invalid_rate': 0.0, 'seconds': 77.48085713386536, 'samples_per_second': 2.581282750324453, 'label_id_style': 'fixed_manual'}
Clearing model from memory: gemma_e2b_it
Loading model once: gemma_e4b_it_4bit_mlx -> 

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

ValueError: Received 126 parameters not in model: 
language_model.model.layers.24.self_attn.k_norm.weight,
language_model.model.layers.24.self_attn.k_proj.biases,
language_model.model.layers.24.self_attn.k_proj.scales,
language_model.model.layers.24.self_attn.k_proj.weight,
language_model.model.layers.24.self_attn.v_proj.biases,
language_model.model.layers.24.self_attn.v_proj.scales,
language_model.model.layers.24.self_attn.v_proj.weight,
language_model.model.layers.25.self_attn.k_norm.weight,
language_model.model.layers.25.self_attn.k_proj.biases,
language_model.model.layers.25.self_attn.k_proj.scales,
language_model.model.layers.25.self_attn.k_proj.weight,
language_model.model.layers.25.self_attn.v_proj.biases,
language_model.model.layers.25.self_attn.v_proj.scales,
language_model.model.layers.25.self_attn.v_proj.weight,
language_model.model.layers.26.self_attn.k_norm.weight,
language_model.model.layers.26.self_attn.k_proj.biases,
language_model.model.layers.26.self_attn.k_proj.scale

In [ ]:
# Inspect invalid examples, if any.
invalids = predictions_df[predictions_df['pred_id'] == '__invalid__']
print('Invalid count:', len(invalids))
invalids.head(20)

## Optional: LoRA fine-tuning

This is optional and may be slow on a 32GB Mac. Start with one task, such as AFQMC. If memory is tight, lower `max_train_samples`, use batch size 1, and keep `gradient_accumulation_steps` modest.

In [ ]:
def finetune_lora_on_task(
    model_key: str,
    model_cfg_or_id,
    task: str = 'afqmc',
    output_dir: Optional[str] = None,
    max_train_samples: int = 1000,
    num_train_epochs: float = 1.0,
):
    if not PEFT_AVAILABLE:
        raise RuntimeError(f'PEFT/TRL are unavailable: {PEFT_IMPORT_ERROR}')

    model_id = get_model_id(model_cfg_or_id)
    backend = get_backend(model_cfg_or_id)
    if backend != 'transformers':
        raise ValueError('This PEFT/TRL LoRA helper only supports Transformers models. MLX LoRA would need a separate MLX training path.')

    if output_dir is None:
        output_dir = f'adapters/{model_key}_{task}_lora'

    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=TORCH_DTYPE,
        device_map=DEVICE_MAP,
        trust_remote_code=True,
    )

    ds = load_dataset('clue', task, split='train')
    label_id_style = update_task_spec_from_dataset(task, ds)
    print(f'{task} labels ({label_id_style}):', TASK_SPECS[task]['label_id_to_name'])
    ds = ds.select(range(min(max_train_samples, len(ds))))
    ds = ds.map(lambda ex: {'text': format_example_for_sft(task, ex)})

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
        task_type='CAUSAL_LM',
    )

    training_args = TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        num_train_epochs=num_train_epochs,
        logging_steps=10,
        save_steps=250,
        save_total_limit=2,
        report_to='none',
        remove_unused_columns=False,
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=ds,
        dataset_text_field='text',
        peft_config=lora_config,
        args=training_args,
        max_seq_length=512,
    )
    trainer.train()
    trainer.save_model(output_dir)
    cleanup_model(model, tokenizer)
    return output_dir


In [ ]:
# Example fine-tuning run. Uncomment one line at a time.
# The MLX quantized Gemma E4B model is inference-only in this notebook.

# qwen_adapter = finetune_lora_on_task('qwen3_4b_instruct_2507', MODELS['qwen3_4b_instruct_2507'], task='afqmc', max_train_samples=1000)
# gemma_e2b_adapter = finetune_lora_on_task('gemma_e2b_it', MODELS['gemma_e2b_it'], task='afqmc', max_train_samples=1000)


In [ ]:
def evaluate_lora_adapter_on_task(
    base_model_cfg_or_id,
    adapter_dir: str,
    model_key: str,
    task: str,
    split: str = 'validation',
    max_samples: Optional[int] = 200,
):
    if not PEFT_AVAILABLE:
        raise RuntimeError(f'PEFT is unavailable: {PEFT_IMPORT_ERROR}')

    base_model_id = get_model_id(base_model_cfg_or_id)
    backend = get_backend(base_model_cfg_or_id)
    if backend != 'transformers':
        raise ValueError('This adapter evaluator only supports Transformers models.')

    dataset = load_dataset('clue', task, split=split)
    label_id_style = update_task_spec_from_dataset(task, dataset)
    print(f'{task} labels ({label_id_style}):', TASK_SPECS[task]['label_id_to_name'])
    if max_samples is not None:
        dataset = dataset.select(range(min(max_samples, len(dataset))))

    tokenizer = AutoTokenizer.from_pretrained(base_model_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(base_model_id, torch_dtype=TORCH_DTYPE, device_map=DEVICE_MAP, trust_remote_code=True)
    model = PeftModel.from_pretrained(base_model, adapter_dir)
    model.eval()

    rows, y_true, y_pred = [], [], []
    start = time.time()
    for ex in tqdm(dataset, desc=f'{model_key}/{task}/adapter'):
        prompt = build_prompt(task, ex)
        raw = generate_answer(tokenizer, model, prompt, max_new_tokens=6, backend='transformers')
        pred_name = extract_label_name(task, raw)
        pred_id = label_name_to_id(task, pred_name)
        gold_id = str(ex['label'])
        gold_name = label_id_to_name(task, gold_id)
        y_true.append(gold_id)
        y_pred.append(pred_id)
        rows.append({'model_key': model_key, 'base_model_id': base_model_id, 'adapter_dir': adapter_dir, 'task': task, 'gold_id': gold_id, 'gold_name': gold_name, 'pred_id': pred_id, 'pred_name': pred_name, 'raw_output': raw})

    seconds = time.time() - start
    summary = {
        'model_key': model_key,
        'base_model_id': base_model_id,
        'adapter_dir': adapter_dir,
        'task': task,
        'split': split,
        'samples': len(y_true),
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'invalid_rate': sum(p == '__invalid__' for p in y_pred) / len(y_pred),
        'seconds': seconds,
        'samples_per_second': len(y_true) / seconds if seconds > 0 else None,
    }
    cleanup_model(model, tokenizer)
    return summary, pd.DataFrame(rows)


In [ ]:
# Example adapter evaluation. Uncomment after training.

# summary, preds = evaluate_lora_adapter_on_task(MODELS['qwen3_4b_instruct_2507'], qwen_adapter, 'qwen3_4b_instruct_2507_lora', 'afqmc')
# print(summary)
# preds.head()
